In [125]:
import pandas as pd

In [126]:
df = pd.read_csv("/content/Resume.csv")


eda starts here inspecting the data


In [127]:
df.shape

(2484, 4)

In [128]:
df.columns

Index(['ID', 'Resume_str', 'Resume_html', 'Category'], dtype='object')

In [129]:
df = df[['Resume_str', 'Category']]

In [130]:
df.isnull().sum()

,0
Resume_str,0
Category,0


In [131]:
category_counts = df['Category'].value_counts()

valid_categories = category_counts[category_counts >= 80].index

df = df[df['Category'].isin(valid_categories)]

In [132]:
df['Category'].value_counts()

,count
Category,
INFORMATION-TECHNOLOGY,120
BUSINESS-DEVELOPMENT,120
ADVOCATE,118
FINANCE,118
CHEF,118
ACCOUNTANT,118
ENGINEERING,118
FITNESS,117
AVIATION,117


In [133]:
df = df.drop_duplicates()

In [134]:
df = df[df['Category'] != 'ENGINEERING']

text preprocessing

In [135]:
import re

imports regular expression which helps in cleaning

In [136]:
def clean_resume(text):

    text = re.sub(r'http\\S+', ' ', text)

    text = re.sub(r'RT|cc', ' ', text)

    text = re.sub(r'#\\S+', '', text)

    text = re.sub(r'@\\S+', '  ', text)

    text = re.sub(r'[^A-Za-z0-9 ]', ' ', text)

    text = re.sub(r'\\s+', ' ', text)

    return text.lower()

removes url and punctuation and coverts in lowercase


In [137]:
df['cleaned_resume'] = df['Resume_str'].apply(clean_resume)

cleans text and stores in new coloumns

In [138]:
df[['Resume_str', 'cleaned_resume']].head()

,Resume_str,cleaned_resume
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,hr administrator marketing associate ...
1,"HR SPECIALIST, US HR OPERATIONS ...",hr specialist us hr operations ...
2,HR DIRECTOR Summary Over 2...,hr director summary over 2...
3,HR SPECIALIST Summary Dedica...,hr specialist summary dedica...
4,HR MANAGER Skill Highlights ...,hr manager skill highlights ...


as we can see text has been cleaned and new text is generated

NOW WE WILL IMPORT TF-IDF VECTORS BECAUSE MODELS UNDERSTAND VECTORS ONLY

In [139]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [140]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=15000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)



this creates the vectorizer

In [141]:
X = tfidf.fit_transform(df['cleaned_resume'])

transformed human language into machine readable vectors fit= learns patterns and transforms= convert text into numbers

In [142]:
y = df['Category']

stores correct output categories

NOW TRAIN TEST SPLIT THis WHERE WE PREPARE DATA FOR LEARNING. WE TRAIN MODEL ON UNSEEN RESUMES OTHERWISE IT WILL MEMORISE IT ALL SO WE DONT TRAIN ON ALL THE DATA.

Training Data
→ AI learns patterns

Testing Data
→ checks if AI truly learned

In [143]:
from sklearn.model_selection import train_test_split

WHAT THIS DOES

Imports:

train_test_split

from Scikit-learn

In [144]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Splits data into:


X_train	:training resume vectors
X_test:	testing resume vectors
y_train:	training labels
y_test	:testing labels
 test size = 0.2 meand 20% data reserved for testing rest for training

random_state=42

Means:

Keep results reproducible

So every run gives same split.

MODEL TRAINING :  we will use logistic regression

In [145]:
from sklearn.linear_model import LogisticRegression

imports Logistic Regression Model

from Scikit-learn

The model is basically:

A mathematical pattern learner

It learns:

Which words relate to which category

In [146]:
model = LogisticRegression(
    max_iter=2000,
    class_weight='balanced'
)

creates empty ai model right now it knows nothing

In [147]:
model.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=2000)


WHAT HAPPENS INTERNALLY

The model analyzes:

resume vectors
categories
word patterns

and learns relationships.

this is supervised learning we give the model correct answers during training.

NOW WE TEST ACCURACY HOW GOOD THE AI IS .

In [148]:
y_pred = model.predict(X_test)

what this does is model looks at unseen resumes and predict categories

In [149]:
from sklearn.metrics import accuracy_score

imports:

Accuracy Calculator

from Scikit-learn

In [150]:
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

0.6904231625835189


compares real categories with predicted categories

In [151]:
from sklearn.metrics import classification_report


In [152]:
print(classification_report(y_test, y_pred))


                        precision    recall  f1-score   support

            ACCOUNTANT       0.65      0.83      0.73        24
              ADVOCATE       0.58      0.46      0.51        24
               APPAREL       0.80      0.42      0.55        19
                  ARTS       0.45      0.24      0.31        21
              AVIATION       0.73      0.96      0.83        23
               BANKING       0.77      0.74      0.76        23
  BUSINESS-DEVELOPMENT       0.59      0.71      0.64        24
                  CHEF       0.86      0.75      0.80        24
          CONSTRUCTION       0.82      0.82      0.82        22
            CONSULTANT       0.78      0.30      0.44        23
              DESIGNER       0.95      1.00      0.98        21
         DIGITAL-MEDIA       0.75      0.79      0.77        19
               FINANCE       0.84      0.67      0.74        24
               FITNESS       0.93      0.58      0.72        24
            HEALTHCARE       0.35      

Evaluation Metrics:

Precision:
Precision measures how many predicted results were actually correct. High precision means the model makes fewer false predictions for a category.

Recall:
Recall measures how many actual instances of a category were correctly identified by the model. High recall means the model successfully identifies most relevant cases.

F1-Score:
F1-score is the balance between precision and recall. It provides an overall measure of classification performance.

Support:
Support represents the number of actual samples available for each category in the testing dataset.

Accuracy:
Accuracy measures the overall correctness of the machine learning model by comparing actual categories with predicted categories. It represents the percentage of correctly classified resumes.

SAVING THE TRAINED MODEL AND VECTORIZED FILES

In [153]:
import joblib

In [154]:
joblib.dump(model, 'model.pkl')

['model.pkl']

In [155]:
joblib.dump(tfidf, 'vectorizer.pkl')

['vectorizer.pkl']

In [156]:
import os

print(os.listdir())

['.config', 'Resume.csv', 'vectorizer.pkl', 'model.pkl', 'sample_data']
